# Lecture 2: Manipulating and combining data

_Please sign attendance sheet; close devices_


## Feeling overwhlemed?

Reminder that learning to code is like learning a spoken language. It's not obvious, and people will pick it up at different speeds at different spots. Try:

- Taking notes in the lecture notebooks
- Using [another Python/pandas learning resource](https://python-public-policy.afeld.me/en/{{school_slug}}/resources.html)
  - Hear things explained another way
  - Ask in [Ed Discussion]({{discussions_url}}) if others have recommendations
- [Comment-driven development](https://www.sitepoint.com/comment-driven-development/)
  - Otherwise, trying to do two steps in your head:
    1. Figuring out the logic
    1. Figuring out the syntax


Small example of comment-driven development:

```python
# find valid ZIP codes
# filter the DataFrame to only invalid ZIP codes
```


## [Boolean indexing](https://pandas.pydata.org/docs/user_guide/10min.html#boolean-indexing)


In [ ]:
import pandas as pd

people = pd.DataFrame(
    {
        "names": ["Fred", "Krishna", "Saron", "Sally"],
        "age": [50, 13, 45, 22],
    }
)
people

In [ ]:
people["age"]

In [ ]:
type(people["age"])

When we compare single values (like `x > 6`), we get a single boolean back. Here, we are checking a _bunch_ of values, so we're going to get multiple booleans, returned as a Series.


In [ ]:
over_40 = people["age"] > 40
over_40

In [ ]:
type(over_40)

In [ ]:
people[over_40]

Can also stuff into one line:

```python
people[people["age"] > 40]
```


## Data cleaning

> Data Cleansing is a process of removing or fixing incorrect, malformed, incomplete, duplicate, or corrupted data

https://hevodata.com/learn/data-cleansing-a-simplified-guide/


When have you needed to clean data?


What are continuous values?


What are categorical values?


### Things to check for

From [my workshop on data cleaning](https://github.com/afeld/data-cleaning):

- Missing data
  - Empty values
- Bad (junk) values
  - Duplicates
  - Mismatched types/formatting
- Categorical values
  - Uniqueness (cardinality)
  - Value counts
- Continuous values
  - Ranges
  - Spread (distribution)


Notes:

- "Values" in this case can be a single cell (in the spreadsheet sense) or a whole row
- "Missing" or "duplicates" can be columns (Series), tables (DataFrames), rows, or cells


### Data cleaning [mnemonic](https://literaryterms.net/mnemonic/)

- Empty
- Bad
- Unique
- Spread


## Setup


In [ ]:
# Display more rows and columns in the DataFrames
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

### Read our cleaned 311 Service Requests dataset


In [ ]:
url = "https://storage.googleapis.com/python-public-policy2/data/311_requests_2018-19_sample_clean.csv.zip"
requests = pd.read_csv(url)

## Dealing with dtypes

More data cleaning!

![Minion character vacuuming](https://impulsecreative.com/hs-fs/hubfs/cleaning-minion-gif.gif?width=490&name=cleaning-minion-gif.gif)


```
DtypeWarning: Columns (8,20,31,34) have mixed types.
```


In [ ]:
requests.info()

In [ ]:
list(requests["Incident Zip"].unique())

ZIP codes _look_ numeric, but aren't really.


[Read the ZIP codes in as strings.](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html#text-data-types)


In [ ]:
requests2 = pd.read_csv(url, dtype={"Incident Zip": "string"})

We fixed the dtype warning for column 8 (`Incident Zip`).


In [ ]:
list(requests2["Incident Zip"].unique())

### Find invalid ZIP codes

Use a [regular expression (regex)](https://regexone.com/) to [find strings that match a pattern](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html#testing-for-strings-that-match-or-contain-a-pattern):

```
^\d{5}(?:-\d{4})?$
? ? ?  ?        ??? end of string
? ? ?  ?        ?? optional
? ? ?  ?? capture group
? ? ?? count
? ?? numeric/digit character
?? start of string
```

Helpful tools:

- [Regexper](https://regexper.com/#%5E%5Cd%7B5%7D%28%3F%3A-%5Cd%7B4%7D%29%3F%24)
- [regex101](https://regex101.com/)


In [ ]:
# find valid ZIP codes
valid_zips = requests2["Incident Zip"].str.contains(r"^\d{5}(?:-\d{4})?$")
valid_zips

In [ ]:
requests2.shape

In [ ]:
requests2[valid_zips].shape

In [ ]:
# filter the DataFrame to only invalid ZIP codes
invalid_zips = valid_zips == False
invalid_zips

In [ ]:
requests_with_invalid_zips = requests2[invalid_zips]
requests_with_invalid_zips["Incident Zip"]

[Clear](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html#inserting-missing-data) any invalid ZIP codes:


In [ ]:
requests2.loc[invalid_zips, "Incident Zip"] = None

[`.loc[]`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) is used for overwriting a subset of values.


Additonal data cleaning tips:

- Hard part is finding what needs to be done
- Will be specific to your use case
- Document what you did, since it will affect your results


## [In-class exercise](https://python-public-policy.afeld.me/en/{{school_slug}}/lecture_2_exercise.html)


## [Concatenation](https://pandas.pydata.org/docs/user_guide/merging.html#concat)


In [ ]:
revenue_q1 = pd.DataFrame(
    {
        "month": ["Jan", "Feb", "Mar"],
        "revenue": [175589, 189645, 163423],
    }
)
revenue_q1

In [ ]:
revenue_q2 = pd.DataFrame(
    {
        "month": ["Apr", "May", "Jun"],
        "revenue": [14567, 15863, 17999],
    }
)
revenue_q2

In [ ]:
pd.concat([revenue_q1, revenue_q2])

## Simple [merge](https://pandas.pydata.org/docs/user_guide/merging.html#merge)


_I had [Copilot](https://code.visualstudio.com/docs/copilot/overview) generate the DataFrames, so no idea if the numbers are real._


In [ ]:
populations = pd.DataFrame(
    {
        "Country": ["China", "India", "Pakistan"],
        "Population": [1444216107, 1393409038, 220892331],
    }
)

populations

In [ ]:
gdps = pd.DataFrame(
    {
        "Country": ["China", "India", "United States", "Indonesia", "Pakistan"],
        "GDP": [14342903, 2875142, 21433226, 1058393, 263687],
    }
)
gdps

How should we combine them?


In [ ]:
populations

In [ ]:
gdps

To join DataFrames together, we will use the [pandas `.merge()` function](https://pandas.pydata.org/pandas-docs/stable/getting_started/intro_tutorials/08_combine_dataframes.html#join-tables-using-a-common-identifier).

![merge diagram](https://pandas.pydata.org/pandas-docs/stable/_images/08_merge_left.svg)


`.merge()` is comparable to:

- [SQL `JOIN`](https://pandas.pydata.org/pandas-docs/stable/getting_started/comparison/comparison_with_sql.html#join)
- [Spreadsheet `VLOOKUP`](https://pandas.pydata.org/pandas-docs/stable/getting_started/comparison/comparison_with_spreadsheets.html#merging)

In general, called ["record linkage" or "entity resolution"](https://en.wikipedia.org/wiki/Record_linkage).


In [ ]:
pd.merge(populations, gdps, on="Country")

In [ ]:
pd.merge(populations, gdps, on="Country", how="outer")

[Different types of merges](https://www.geeksforgeeks.org/different-types-of-joins-in-pandas/)


## In-class exercise 2

Compute the migrant population as a percent of total by country using [UN data](https://data.un.org/). [Pair](https://github.com/advanced-computing/course-materials/blob/main/docs/pairing.md) with a neighbor.


## [Homework 2](https://python-public-policy.afeld.me/en/{{school_slug}}/hw_2.html)
